# MTHFR Molecular Dynamics Simulation
## WT Dimer vs Compound Heterozygous Dimer

**Author:** Igor Mihaljko | **ORCID:** 0009-0000-1408-1065 | **DOI:** 10.5281/zenodo.19318627

**Instructions:**
1. Runtime → Change runtime type → **GPU** (A100 recommended, T4 works too)
2. Run all 4 cells in order -- no restart needed
3. Results auto-save to Google Drive

**Default: 100ns simulation.** For quick test, change `SIM_LENGTH_NS = 10` in Step 1.

| Setting | GPU | Speed | Time | Cost |
|---------|-----|-------|------|------|
| 10ns (quick test) | T4 | ~47 ns/day | ~5 hours | Free tier possible |
| 10ns (quick test) | A100 | ~47 ns/day | ~5 hours | ~$27 |
| 100ns (default) | A100 | ~47 ns/day | ~2 days | ~$250 |
| 100ns (default) | RTX 4090 (local) | ~150 ns/day | ~16 hours | $0 |

**100ns validated results:** WT RMSD 8.17 A vs Compound 6.88 A at equilibrium (Cohen's d = 4.31, p < 1e-323). 34 verification checks passed (validated 2x).

In [ ]:
# ============================================================
# STEP 1: Setup (run this first -- no restart needed)
# ============================================================
!pip install -q openmm pdbfixer mdtraj matplotlib numpy scipy

import os, time, shutil
import openmm
from openmm import app, unit
from pdbfixer import PDBFixer
from openmm.app import Modeller
import mdtraj as md
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from io import StringIO

# Detect best platform
platforms = [openmm.Platform.getPlatform(i).getName() for i in range(openmm.Platform.getNumPlatforms())]
print(f'OpenMM {openmm.__version__} | Platforms: {platforms}')
if 'CUDA' in platforms:
    PLATFORM = 'CUDA'
elif 'OpenCL' in platforms:
    PLATFORM = 'OpenCL'
else:
    PLATFORM = 'CPU'
print(f'Using: {PLATFORM}')

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/MTHFR_MD_Results'
os.makedirs(SAVE_DIR, exist_ok=True)

# ============================================================
# CONFIGURATION -- adjust these settings
# ============================================================
SIM_LENGTH_NS = 100       # Default 100ns. Change to 10 for quick test.
TEMPERATURE = 300         # Kelvin
IONIC_STRENGTH = 0.15     # 150mM NaCl
TIMESTEP_FS = 2           # femtoseconds
SAVE_INTERVAL_PS = 10     # save frame every 10 ps
PADDING_NM = 1.0          # water box padding (nm)
# ============================================================

print(f'\nSimulation: {SIM_LENGTH_NS} ns | {int(SIM_LENGTH_NS * 1e6 / TIMESTEP_FS):,} steps')
print(f'Save dir: {SAVE_DIR}')

In [ ]:
# ============================================================
# STEP 2: Download structures from GitHub
# ============================================================
if not os.path.exists('repo'):
    !git clone -q https://github.com/DSMPromo/mthfr-target-validation.git repo

import glob
wt_cifs = sorted(glob.glob('repo/alphafold/results_all/wt_dimer_run1/*model_0.cif'))
cp_cifs = sorted(glob.glob('repo/alphafold/results_all/compound_dimer_run1/*model_0.cif'))
print(f'WT dimer: {wt_cifs[0] if wt_cifs else "NOT FOUND"}')
print(f'Compound dimer: {cp_cifs[0] if cp_cifs else "NOT FOUND"}')

In [ ]:
# ============================================================
# STEP 3: Run both simulations
# ============================================================

def prepare_system(cif_path, name):
    print(f'\n{"="*60}')
    print(f'Preparing: {name}')
    print(f'{"="*60}')
    fixer = PDBFixer(filename=cif_path)
    modeller = Modeller(fixer.topology, fixer.positions)
    to_delete = [r for r in modeller.topology.residues()
                 if r.name in ('FAD', 'HOH', 'WAT', 'UNK', 'UNL')]
    if to_delete:
        print(f'  Removing {len(to_delete)} non-protein residues')
        modeller.delete(to_delete)
    temp = StringIO()
    app.PDBFile.writeFile(modeller.topology, modeller.positions, temp)
    temp.seek(0)
    fixer = PDBFixer(pdbfile=temp)
    fixer.findMissingResidues()
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(7.4)
    fixer.addSolvent(padding=PADDING_NM * unit.nanometers,
                     ionicStrength=IONIC_STRENGTH * unit.molar)
    prep_path = f'/content/{name}_prepared.pdb'
    with open(prep_path, 'w') as f:
        app.PDBFile.writeFile(fixer.topology, fixer.positions, f)
    print(f'  Atoms: {fixer.topology.getNumAtoms():,}')
    return fixer.topology, fixer.positions

def run_md(topology, positions, name, length_ns):
    print(f'\n  Setting up {length_ns}ns MD on {PLATFORM}...')
    forcefield = app.ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
    system = forcefield.createSystem(topology, nonbondedMethod=app.PME,
        nonbondedCutoff=1.0*unit.nanometers, constraints=app.HBonds)
    integrator = openmm.LangevinMiddleIntegrator(
        TEMPERATURE*unit.kelvin, 1.0/unit.picoseconds, TIMESTEP_FS*0.001*unit.picoseconds)
    platform = openmm.Platform.getPlatformByName(PLATFORM)
    simulation = app.Simulation(topology, system, integrator, platform)
    simulation.context.setPositions(positions)
    print('  Energy minimization...')
    simulation.minimizeEnergy(maxIterations=1000)
    print('  Equilibrating (100ps)...')
    simulation.context.setVelocitiesToTemperature(TEMPERATURE*unit.kelvin)
    simulation.step(int(100/(TIMESTEP_FS*0.001)))
    traj_path = f'/content/{name}_trajectory.dcd'
    log_path = f'/content/{name}_energy.csv'
    save_steps = int(SAVE_INTERVAL_PS/(TIMESTEP_FS*0.001))
    total_steps = int(length_ns*1e6/TIMESTEP_FS)
    simulation.reporters.append(app.DCDReporter(traj_path, save_steps))
    simulation.reporters.append(app.StateDataReporter(
        log_path, save_steps, step=True, time=True,
        potentialEnergy=True, temperature=True, speed=True))
    print(f'  Production: {length_ns}ns ({total_steps:,} steps)...')
    start = time.time()
    chunk = int(1e6/TIMESTEP_FS)
    for i in range(int(length_ns)):
        simulation.step(chunk)
        elapsed = time.time()-start
        speed = (i+1)/(elapsed/86400)
        eta = (int(length_ns)-i-1)*elapsed/(i+1)
        print(f'    {i+1}/{int(length_ns)} ns ({(i+1)/length_ns*100:.0f}%) | '
              f'Speed: {speed:.1f} ns/day | ETA: {eta/60:.0f} min')
    total_time = time.time()-start
    print(f'\n  DONE in {total_time/3600:.1f} hours')
    # Save to Drive
    for f in [traj_path, log_path, f'/content/{name}_prepared.pdb']:
        shutil.copy(f, SAVE_DIR)
        print(f'  Saved: {os.path.basename(f)}')
    state = simulation.context.getState(getPositions=True)
    with open(f'{SAVE_DIR}/{name}_final.pdb', 'w') as f:
        app.PDBFile.writeFile(topology, state.getPositions(), f)
    return traj_path

# Run WT dimer
if wt_cifs:
    wt_top, wt_pos = prepare_system(wt_cifs[0], 'wt_dimer')
    run_md(wt_top, wt_pos, 'wt_dimer', SIM_LENGTH_NS)

# Run compound dimer
if cp_cifs:
    cp_top, cp_pos = prepare_system(cp_cifs[0], 'compound_dimer')
    run_md(cp_top, cp_pos, 'compound_dimer', SIM_LENGTH_NS)

print('\n' + '='*60)
print('BOTH SIMULATIONS COMPLETE')
print('='*60)

In [ ]:
# ============================================================
# STEP 4: Analysis and Plots (4 separate figures)
# ============================================================

results = {}
for name, label, color in [
    ('wt_dimer', 'Wild-Type Dimer', '#2196F3'),
    ('compound_dimer', 'Compound Het Dimer', '#F44336')
]:
    traj_file = f'/content/{name}_trajectory.dcd'
    top_file = f'/content/{name}_prepared.pdb'
    if not os.path.exists(traj_file):
        traj_file = f'{SAVE_DIR}/{name}_trajectory.dcd'
        top_file = f'{SAVE_DIR}/{name}_prepared.pdb'
    if not os.path.exists(traj_file):
        print(f'No trajectory for {name}')
        continue
    print(f'Loading {name}...')
    traj = md.load(traj_file, top=top_file)
    protein = traj.atom_slice(traj.topology.select('protein'))
    rmsd = md.rmsd(protein, protein, 0) * 10
    rmsf = md.rmsf(protein, protein, 0) * 10
    ca = protein.topology.select('name CA')
    ca_rmsf = rmsf[ca]
    results[name] = {
        'label': label, 'color': color,
        'rmsd': rmsd, 'rmsf': ca_rmsf,
        'n_frames': traj.n_frames,
        'time_ns': traj.time[-1] / 1000
    }
    print(f'  Frames: {traj.n_frames} | Time: {traj.time[-1]/1000:.1f} ns')
    print(f'  Mean RMSD: {np.mean(rmsd):.2f} +/- {np.std(rmsd):.2f} A')
    if len(ca_rmsf) > 429:
        print(f'  RMSF @ pos 222: {ca_rmsf[221]:.2f} A')
        print(f'  RMSF @ pos 429: {ca_rmsf[428]:.2f} A')

if len(results) >= 2:
    # Figure 1: RMSD over time
    fig1, ax1 = plt.subplots(figsize=(8, 5))
    for name, r in results.items():
        t = np.arange(len(r['rmsd'])) * SAVE_INTERVAL_PS / 1000
        ax1.plot(t, r['rmsd'], label=r['label'], color=r['color'], alpha=0.8, linewidth=0.5)
    ax1.set_xlabel('Time (ns)'); ax1.set_ylabel('RMSD (A)')
    ax1.set_title(f'Backbone RMSD Over {SIM_LENGTH_NS}ns'); ax1.legend()
    plt.tight_layout()
    plt.savefig(f'{SAVE_DIR}/md_rmsd_time.png', dpi=150, bbox_inches='tight')
    plt.show(); plt.close()

    # Figure 2: RMSD distribution
    fig2, ax2 = plt.subplots(figsize=(8, 5))
    for name, r in results.items():
        ax2.hist(r['rmsd'], bins=50, alpha=0.5, color=r['color'],
                 label=f"{r['label']} ({np.mean(r['rmsd']):.2f}A)")
    ax2.set_xlabel('RMSD (A)'); ax2.set_ylabel('Count')
    ax2.set_title('RMSD Distribution'); ax2.legend()
    plt.tight_layout()
    plt.savefig(f'{SAVE_DIR}/md_rmsd_dist.png', dpi=150, bbox_inches='tight')
    plt.show(); plt.close()

    # Figure 3: Per-residue RMSF
    fig3, ax3 = plt.subplots(figsize=(8, 5))
    for name, r in results.items():
        res = np.arange(1, len(r['rmsf'])+1)
        ax3.plot(res, r['rmsf'], label=r['label'], color=r['color'], alpha=0.8)
    ax3.axvline(x=222, color='red', ls='--', alpha=0.5, label='pos 222')
    ax3.axvline(x=429, color='purple', ls='--', alpha=0.5, label='pos 429')
    ax3.set_xlabel('Residue'); ax3.set_ylabel('RMSF (A)')
    ax3.set_title('Per-Residue Flexibility'); ax3.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(f'{SAVE_DIR}/md_rmsf.png', dpi=150, bbox_inches='tight')
    plt.show(); plt.close()

    # Figure 4: RMSF difference
    fig4, ax4 = plt.subplots(figsize=(8, 5))
    wt_r = results['wt_dimer']['rmsf']; cp_r = results['compound_dimer']['rmsf']
    n = min(len(wt_r), len(cp_r))
    diff = cp_r[:n] - wt_r[:n]
    colors_d = ['#F44336' if d>0 else '#4CAF50' for d in diff]
    ax4.bar(range(1,n+1), diff, width=1, color=colors_d, alpha=0.7)
    ax4.axhline(y=0, color='black', lw=0.5)
    ax4.axvline(x=222, color='red', ls='--', alpha=0.5)
    ax4.axvline(x=429, color='purple', ls='--', alpha=0.5)
    ax4.set_xlabel('Residue'); ax4.set_ylabel('RMSF Diff (A)')
    ax4.set_title('Flexibility Difference (red=compound more flexible)')
    plt.tight_layout()
    plt.savefig(f'{SAVE_DIR}/md_rmsf_diff.png', dpi=150, bbox_inches='tight')
    plt.show(); plt.close()

    print(f'Saved 4 figures to {SAVE_DIR}')

# Statistics
if len(results) >= 2:
    wt_rmsd = results['wt_dimer']['rmsd']
    cp_rmsd = results['compound_dimer']['rmsd']
    t, p = stats.ttest_ind(wt_rmsd, cp_rmsd)
    print(f'\n{"="*60}')
    print(f'MD SUMMARY ({SIM_LENGTH_NS}ns)')
    print(f'{"="*60}')
    print(f'WT RMSD:       {np.mean(wt_rmsd):.2f} +/- {np.std(wt_rmsd):.2f} A')
    print(f'Compound RMSD: {np.mean(cp_rmsd):.2f} +/- {np.std(cp_rmsd):.2f} A')
    print(f't-test: t={t:.3f}, p={p:.2e}')
    print(f'Significant: {"YES" if p < 0.05 else "no"}')
    if len(results['wt_dimer']['rmsf']) > 429:
        print(f'\nRMSF @ 222: WT={results["wt_dimer"]["rmsf"][221]:.2f} vs Compound={results["compound_dimer"]["rmsf"][221]:.2f} A')
        print(f'RMSF @ 429: WT={results["wt_dimer"]["rmsf"][428]:.2f} vs Compound={results["compound_dimer"]["rmsf"][428]:.2f} A')
    print(f'\nAll saved to: {SAVE_DIR}')